# Three leaks, made visible

This is the methodological contribution of the E01/E02 investigation: three separate leaks were
found and fixed, and each is small enough to state as a single number. Reads
`../data/training_set.parquet` and `../experiments/csv/*.csv` only.

## 1. Setup

In [1]:
import pandas as pd, numpy as np
from argotech.lab.estimand.peers import peer_key, fit_peer_stats, apply_peer_z
from argotech.lab.eval.splits import leave_one_cluster_out

panel = pd.read_parquet('../data/training_set.parquet')
panel['obs_date'] = pd.to_datetime(panel['obs_date'])
panel['label_date'] = pd.to_datetime(panel['label_date'])
CLUSTER_ORDER = ["Benue_River_Basin", "Kaduna_Grain_Belt", "Kano_Sudan_Savannah", "Kenya_Rift_Valley"]
panel.shape

(4596, 35)

## 2. Leak 1 — the peer reference, fitted per fold

`ndvi_z_peer`/`rvi_z_peer` used to be a whole-frame column: a held-out cluster's own bucket
contributed to its own reference. Fit per fold instead (`fit_peer_stats` sees training rows only),
an honest `cluster_month` key gives a held-out cluster **no reference at all** -- its own bucket
never appeared in training. A geographic key (`geo_month`, fixed lat/elevation bands) restores
coverage because the band an unseen cluster falls in is also populated by other clusters.

In [2]:
for kind in ('cluster_month', 'geo_month'):
    coverage = []
    for cluster, train, test in leave_one_cluster_out(panel):
        stats = fit_peer_stats(train, kind)
        applied, _ = apply_peer_z(panel, stats, kind)
        test_z = applied.loc[applied.index.isin(test.index), 'ndvi_z_peer']
        coverage.append(round(float(test_z.notna().mean()) if len(test_z) else 0.0, 3))
    print(f"{kind:14s} coverage by held-out cluster {CLUSTER_ORDER}: {coverage}")

cluster_month  coverage by held-out cluster ['Benue_River_Basin', 'Kaduna_Grain_Belt', 'Kano_Sudan_Savannah', 'Kenya_Rift_Valley']: [0.0, 0.0, 0.0, 0.0]
geo_month      coverage by held-out cluster ['Benue_River_Basin', 'Kaduna_Grain_Belt', 'Kano_Sudan_Savannah', 'Kenya_Rift_Valley']: [1.0, 1.0, 1.0, 0.472]


## 3. Leak 2 — the temporal cut

An earlier version of the evaluation cut *training* rows on `obs_date` (the prediction date)
instead of `label_date` (the date the outcome became known), and a later commit re-shipped the same
defect under a message claiming to fix it. The lag is exactly 30 days on every row (§Task 2), so this
is checkable directly: pick a boundary, and count how many "training" rows under each cut have an
outcome that postdates the first test prediction.

In [3]:
boundary = pd.to_datetime(panel.obs_date).quantile(0.65)
print(f"boundary: {boundary.date()}")

train_on_obs = panel[panel.obs_date < boundary]
leaked = (train_on_obs.label_date >= boundary).sum()
print(f"cut on obs_date   : {len(train_on_obs)} training rows, {leaked} with an outcome that "
      f"postdates the first test prediction")

train_on_label = panel[panel.label_date < boundary]
leaked2 = (train_on_label.label_date >= boundary).sum()
print(f"cut on label_date : {len(train_on_label)} training rows, {leaked2} postdating")

boundary: 2025-03-28
cut on obs_date   : 2931 training rows, 121 with an outcome that postdates the first test prediction
cut on label_date : 2810 training rows, 0 postdating


Cutting on `obs_date` puts 121 rows in training whose own label was not yet knowable at the
first test prediction -- a straightforward leak. Cutting on `label_date` (what
`lab.eval.splits.forward_chaining` does today) admits zero, by construction: every training row's
outcome is required to already be in the past.

## 4. Leak 3 — the band trade-off

`geo_month`'s lat/elevation bands trade transfer against specificity. A wider band gives a held-out
cluster more donor rows (training rows sharing its bucket) but a less locally-relevant reference; a
narrower band is the reverse. "Donor rows" is a coverage-mechanism count, computed the same way
`experiments/E04-band-sweep/band_sweep.py`'s own anchors were validated: peer-key bucket membership,
no model fit.

In [4]:
def donor_rows(df, lat_band, elev_band):
    out = {}
    for cluster, train, test in leave_one_cluster_out(df):
        train_keys = peer_key(train, 'geo_month', lat_band=lat_band, elev_band=elev_band)
        test_keys = peer_key(test, 'geo_month', lat_band=lat_band, elev_band=elev_band)
        out[cluster] = int(train_keys.isin(set(test_keys.unique())).sum())
    return out

for lat_band, elev_band in [(5.0, 500.0), (10.0, 1000.0), (20.0, 1000.0)]:
    d = donor_rows(panel, lat_band, elev_band)
    print(f"{lat_band:>5.0f} deg / {elev_band:>5.0f} m : {[d[c] for c in CLUSTER_ORDER]}")

    5 deg /   500 m : [0, 282, 1098, 0]
   10 deg /  1000 m : [36, 1353, 1098, 647]
   20 deg /  1000 m : [2175, 2799, 1792, 3139]


At 5°/500 m two clusters get zero donors at all -- too narrow to transfer. At 20°/1000 m every
cluster gets thousands -- wide enough that "geographic" band starts to mean "everywhere". 10°/1000 m
is the point actually used elsewhere in this investigation: every cluster gets at least some donors,
none gets the whole panel.

## 5. Closing the loop: multiple comparisons

E02-E05 tested many (target, peer key, split, arm) cells. Reading any single positive result off that
sweep without accounting for how many were tried is the fourth way to fool yourself, after the three
leaks above. `experiments/csv/cell_summary.csv` is the committed record of every cell actually run.

In [5]:
cells = pd.read_csv('../experiments/csv/cell_summary.csv')

for split, expected_frac in [('spatial', 0.025), ('temporal', 0.025)]:
    sub = cells[cells.split == split]
    tested = len(sub)
    sig_pos = int(((sub.ci_excludes_zero == True) & (sub.net_benefit_mean > 0)).sum())  # noqa: E712
    expected = expected_frac * tested
    print(f"{split:9s}: {tested} tested / {sig_pos} significant-positive / "
          f"~{expected:.1f} expected by chance at alpha=0.05 one-sided "
          f"({sig_pos/expected:.1f}x)")

spatial  : 165 tested / 8 significant-positive / ~4.1 expected by chance at alpha=0.05 one-sided (1.9x)
temporal : 65 tested / 13 significant-positive / ~1.6 expected by chance at alpha=0.05 one-sided (8.0x)


**Spatial**: 8 of 165 clear zero against ~4.1 expected by chance -- a 1.9x enrichment, weak
evidence of real (if narrow) transfer. **Temporal**: 13 of 65 clear zero against ~1.6 expected -- an
8.0x enrichment, much stronger. Read together with the leaks above: a positive cell is worth taking
seriously only after its reference was fit honestly (§2), its split was cut on the right date (§3),
and its band was wide enough to have donors at all (§4) -- and even then, only in proportion to how
many cells were tried to find it.